# Sistema Inteligente para Priorização de Chamados de Manutenção

**Engenharia de Computação · Autor e único integrante: Guilherme Costa**  
**Prazo da entrega: 20/11/2026 · Idioma: português do Brasil**

Protótipo para escolas, ONGs e pequenas instituições. A árvore sugere **0 = normal** ou **1 = prioritário**. O responsável pela manutenção revisa os campos e decide a ordem de atendimento.

**Origem: todos os dados deste notebook são fictícios.** Nenhum chamado real foi coletado. O notebook gera sua própria base e funciona em uma sessão limpa, sem importar arquivos do repositório. Esta cópia foi executada e teve suas saídas verificadas em **ambiente local Codex Cloud com Python e nbclient**; executar aqui não comprova execução no Google Colab.

No Colab: abra `https://colab.research.google.com/`, escolha **Arquivo → Fazer upload de notebook**, envie este `.ipynb` e use **Ambiente de execução → Executar tudo**. Salve uma cópia para preservar as novas saídas. A primeira célula instala versões compatíveis somente se necessário.

## 1. Dependências e ambiente

As versões de pandas, scikit-learn e dependências numéricas são fixadas para reproduzir o treinamento. A instalação precisa de internet quando as versões não estiverem disponíveis; não são necessárias credenciais, API externa ou arquivos anexados. O restante do notebook não depende de rede.

In [1]:
import importlib.metadata as metadata
import subprocess
import sys
pacotes = {"numpy": "2.3.5", "scipy": "1.17.0", "pandas": "2.2.3", "scikit-learn": "1.8.0", "joblib": "1.5.3"}
faltantes = []
for pacote, versao in pacotes.items():
    try:
        if metadata.version(pacote) != versao:
            faltantes.append(f"{pacote}=={versao}")
    except metadata.PackageNotFoundError:
        faltantes.append(f"{pacote}=={versao}")
if faltantes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *faltantes])
print("Dependências verificadas:", pacotes)
print("Python:", sys.version.split()[0])

Dependências verificadas: {'numpy': '2.3.5', 'scipy': '1.17.0', 'pandas': '2.2.3', 'scikit-learn': '1.8.0', 'joblib': '1.5.3'}
Python: 3.12.14


In [2]:
from pathlib import Path
import json
import platform
import os
import sys
import hashlib
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, recall_score, precision_score,
                             confusion_matrix, make_scorer, fbeta_score)

FEATURES = ["risco_seguranca", "servico_essencial_interrompido", "pessoas_afetadas",
            "dias_aguardando", "recorrente"]
SEMENTE = 42
ORIGEM = "Base inteiramente fictícia, criada para demonstração acadêmica; nenhum chamado real."



## 2. Política de rotulagem anterior ao treino

Os limiares abaixo são pedagógicos. Não foram escolhidos por resultado no teste e não são normas de manutenção. Chamados urgentes de segurança continuam prioritários mesmo com zero pessoas presentes no momento.

In [3]:
POLITICA = """Política didática definida ANTES do treino: prioritário (1) se houver
risco à segurança OU interrupção de serviço essencial OU (pelo menos 40 pessoas
afetadas E pelo menos 7 dias de espera) OU (recorrência E pelo menos 10 pessoas
afetadas E pelo menos 14 dias de espera). Nos demais casos, normal (0).
Segurança independe do número de pessoas: fiação exposta em local pouco usado
continua prioritária. Os limiares de espera/impacto são escolhas pedagógicas,
não padrões técnicos universais. Em uso real devem ser pactuados com a instituição.
Serviço essencial significa indisponibilidade de água, energia, saneamento,
alimentação ou acesso indispensável; equipamento auxiliar com alternativa não
é automaticamente essencial. O responsável sempre revisa a sugestão."""


def rotular(valores):
    risco, essencial, pessoas, dias, recorrente = valores
    motivos = []
    if risco:
        motivos.append("Risco à segurança, mesmo com pouca circulação")
    if essencial:
        motivos.append("Serviço essencial interrompido")
    if pessoas >= 40 and dias >= 7:
        motivos.append("Impacto em pelo menos 40 pessoas e espera de pelo menos 7 dias")
    if recorrente and pessoas >= 10 and dias >= 14:
        motivos.append("Recorrência, pelo menos 10 pessoas e espera de pelo menos 14 dias")
    return int(bool(motivos)), "; ".join(motivos) or "Baixo impacto segundo a política didática"


print(POLITICA)

Política didática definida ANTES do treino: prioritário (1) se houver
risco à segurança OU interrupção de serviço essencial OU (pelo menos 40 pessoas
afetadas E pelo menos 7 dias de espera) OU (recorrência E pelo menos 10 pessoas
afetadas E pelo menos 14 dias de espera). Nos demais casos, normal (0).
Segurança independe do número de pessoas: fiação exposta em local pouco usado
continua prioritária. Os limiares de espera/impacto são escolhas pedagógicas,
não padrões técnicos universais. Em uso real devem ser pactuados com a instituição.
Serviço essencial significa indisponibilidade de água, energia, saneamento,
alimentação ou acesso indispensável; equipamento auxiliar com alternativa não
é automaticamente essencial. O responsável sempre revisa a sugestão.


## 3. Base incorporada, reproduzível e validada

São **30 famílias × 4 variações = 120 registros**. Há baixo impacto, risco, interrupção essencial e fronteiras de 39/40 pessoas, 6/7 dias, 9/10 pessoas e 13/14 dias. Família, descrição, ID e justificativa não entram em X. Um valor desconhecido não equivale a zero: o protótipo exige confirmação e preenchimento manual das entradas ausentes.

In [4]:

def gerar_base():
    # Cada família reúne quatro variações do MESMO tipo de cenário. Tuplas em FEATURES.
    cenarios = [
        ("macaneta_sala_vazia", "Maçaneta frouxa em sala desocupada, sem risco de aprisionamento", [(0,0,0,0,0),(0,0,0,2,0),(0,0,0,9,1),(0,0,0,20,1)]),
        ("pintura_corredor", "Pintura descascada sem umidade ou risco no corredor", [(0,0,2,3,0),(0,0,5,10,0),(0,0,10,25,0),(0,0,20,30,0)]),
        ("lampada_deposito", "Lâmpada de depósito falhou; iluminação alternativa suficiente", [(0,0,1,1,0),(0,0,2,4,0),(0,0,4,8,0),(0,0,0,18,1)]),
        ("cadeira_estofado", "Estofado rasgado de cadeira retirada de uso, estrutura íntegra", [(0,0,1,0,0),(0,0,3,2,0),(0,0,2,15,0),(0,0,5,25,1)]),
        ("projetor_reserva", "Projetor auxiliar sem imagem; existe equipamento substituto", [(0,0,0,2,0),(0,0,12,3,1),(0,0,20,5,0),(0,0,25,6,1)]),
        ("persiana_reuniao", "Persiana travada na sala de reunião, sem peças soltas", [(0,0,4,2,0),(0,0,6,7,0),(0,0,8,15,0),(0,0,12,10,1)]),
        ("torneira_auxiliar", "Torneira auxiliar com gotejamento; outras pias disponíveis", [(0,0,3,1,0),(0,0,5,4,1),(0,0,8,15,1),(0,0,9,25,1)]),
        ("impressora_secretaria", "Impressora administrativa indisponível, com alternativa", [(0,0,4,0,0),(0,0,8,6,1),(0,0,10,13,1),(0,0,15,20,0)]),
        ("ventilador_quadra", "Ventilador auxiliar da quadra falhou, ambiente ventilado", [(0,0,15,3,0),(0,0,20,7,0),(0,0,30,9,0),(0,0,35,12,0)]),
        ("placa_identificacao", "Placa de identificação apagada; sinalização de emergência íntegra", [(0,0,5,5,0),(0,0,10,7,0),(0,0,20,10,0),(0,0,39,20,0)]),
        ("internet_leitura", "Rede auxiliar da sala de leitura falhou; atividades essenciais continuam", [(0,0,39,6,0),(0,0,40,6,0),(0,0,41,7,0),(0,0,60,10,0)]),
        ("laboratorio_tomadas", "Tomadas auxiliares desativadas com segurança no laboratório", [(0,0,35,7,0),(0,0,40,7,0),(0,0,45,6,0),(0,0,50,8,0)]),
        ("som_auditorio", "Som auxiliar do auditório falhou, sem interromper serviço essencial", [(0,0,40,6,0),(0,0,40,7,0),(0,0,80,8,1),(0,0,39,8,0)]),
        ("banheiro_recorrente", "Descarga auxiliar requer regulagem recorrente; demais banheiros disponíveis", [(0,0,9,14,1),(0,0,10,14,1),(0,0,10,13,1),(0,0,11,15,1)]),
        ("porta_almoxarifado", "Porta interna arrasta no piso, sem bloquear saída ou acesso essencial", [(0,0,9,13,1),(0,0,10,14,1),(0,0,12,15,1),(0,0,15,16,1)]),
        ("rede_oficina", "Rede auxiliar de oficina oscila repetidamente, com alternativa disponível", [(0,0,10,13,1),(0,0,10,14,1),(0,0,20,14,0),(0,0,20,14,1)]),
        ("fiacao_pouca_circulacao", "Fiação exposta em área de pouca circulação: possível choque elétrico", [(1,0,0,0,0),(1,0,1,0,0),(1,0,2,1,1),(1,0,3,5,0)]),
        ("piso_escorregadio", "Vazamento deixa piso de circulação escorregadio", [(1,0,5,0,0),(1,0,20,1,0),(1,0,40,2,1),(1,0,80,3,1)]),
        ("corrimao_escada", "Corrimão solto em escada utilizada por pessoas", [(1,0,2,0,0),(1,0,15,2,0),(1,0,50,6,1),(1,0,100,8,1)]),
        ("porta_emergencia", "Porta de saída de emergência obstruída", [(1,0,0,0,0),(1,0,30,1,0),(1,0,70,2,1),(1,0,150,3,0)]),
        ("forro_solto", "Placa do forro solta com risco de queda", [(1,0,0,1,0),(1,0,4,2,0),(1,0,20,4,1),(1,0,45,10,1)]),
        ("vazamento_gas", "Odor de gás em cozinha, uso suspenso e área isolada", [(1,1,5,0,0),(1,1,20,0,1),(1,1,80,1,0),(1,1,120,2,1)]),
        ("falta_agua_escola", "Falta de água na escola, impedindo higiene e funcionamento", [(0,1,30,0,0),(0,1,100,1,0),(0,1,200,2,1),(0,1,350,4,1)]),
        ("energia_ong", "Falta de energia interrompe atendimento essencial da ONG", [(0,1,8,0,0),(0,1,20,1,1),(0,1,50,3,0),(0,1,90,7,1)]),
        ("esgoto_interrompido", "Saneamento indisponível e sanitários essenciais interditados", [(1,1,10,0,0),(1,1,25,1,0),(1,1,60,3,1),(1,1,180,5,1)]),
        ("cozinha_sem_refrigeracao", "Refrigeração da alimentação institucional interrompida", [(0,1,15,0,0),(0,1,40,1,0),(0,1,90,2,1),(0,1,220,6,1)]),
        ("elevador_acessivel", "Único acesso acessível aos serviços essenciais está indisponível", [(0,1,1,0,0),(0,1,2,1,1),(0,1,6,4,0),(0,1,12,10,1)]),
        ("armario_auxiliar", "Fechadura de armário auxiliar falha, sem itens essenciais retidos", [(0,0,2,30,1),(0,0,9,14,1),(0,0,10,14,1),(0,0,11,20,1)]),
        ("acustica_oficina", "Painel acústico requer reparo estético, firmemente preso", [(0,0,39,7,0),(0,0,40,7,0),(0,0,41,6,1),(0,0,42,8,1)]),
        ("computadores_auxiliares", "Computadores auxiliares indisponíveis, atendimento essencial mantido", [(0,0,9,20,1),(0,0,10,20,1),(0,0,30,13,1),(0,0,30,14,1)]),
    ]
    registros = []
    for familia, descricao, variacoes in cenarios:
        for i, valores in enumerate(variacoes, start=1):
            classe, justificativa = rotular(valores)
            registros.append({"id": f"FIC-{len(registros)+1:03d}",
                "familia_cenario": familia,
                "descricao": f"{descricao}. Variação fictícia {i}: {valores[2]} pessoa(s) afetada(s), {valores[3]} dia(s) de espera; recorrência {'sim' if valores[4] else 'não'}.",
                **dict(zip(FEATURES, valores)), "prioridade": classe,
                "justificativa_rotulo": justificativa, "origem": "fictícia"})
    return pd.DataFrame(registros)



In [5]:

def validar_base(df):
    assert len(df) == 120, "São esperados 120 chamados fictícios"
    assert df["id"].is_unique and not df.isna().any().any()
    assert df["familia_cenario"].nunique() == 30
    assert df.groupby("familia_cenario").size().eq(4).all()
    for campo in FEATURES + ["prioridade"]:
        assert pd.api.types.is_integer_dtype(df[campo]), campo
        assert (df[campo] >= 0).all(), campo
    for campo in ["risco_seguranca", "servico_essencial_interrompido", "recorrente", "prioridade"]:
        assert set(df[campo].unique()) <= {0, 1}, campo
    assert set(df["prioridade"].unique()) == {0, 1}
    assert df["origem"].eq("fictícia").all()
    assert all(rotular(row)[0] == alvo for row, alvo in zip(df[FEATURES].to_numpy(), df.prioridade))
    return {"registros": len(df), "familias": int(df.familia_cenario.nunique()),
            "classes": {str(k):int(v) for k,v in df.prioridade.value_counts().sort_index().items()}}


df = gerar_base()
print(validar_base(df))
display(df.head(8))
display(df.groupby("prioridade").size().rename("registros").to_frame())

{'registros': 120, 'familias': 30, 'classes': {'0': 57, '1': 63}}


,id,familia_cenario,descricao,risco_seguranca,servico_essencial_interrompido,pessoas_afetadas,dias_aguardando,recorrente,prioridade,justificativa_rotulo,origem
0,FIC-001,macaneta_sala_vazia,"Maçaneta frouxa em sala desocupada, sem risco ...",0,0,0,0,0,0,Baixo impacto segundo a política didática,fictícia
1,FIC-002,macaneta_sala_vazia,"Maçaneta frouxa em sala desocupada, sem risco ...",0,0,0,2,0,0,Baixo impacto segundo a política didática,fictícia
2,FIC-003,macaneta_sala_vazia,"Maçaneta frouxa em sala desocupada, sem risco ...",0,0,0,9,1,0,Baixo impacto segundo a política didática,fictícia
3,FIC-004,macaneta_sala_vazia,"Maçaneta frouxa em sala desocupada, sem risco ...",0,0,0,20,1,0,Baixo impacto segundo a política didática,fictícia
4,FIC-005,pintura_corredor,Pintura descascada sem umidade ou risco no cor...,0,0,2,3,0,0,Baixo impacto segundo a política didática,fictícia
5,FIC-006,pintura_corredor,Pintura descascada sem umidade ou risco no cor...,0,0,5,10,0,0,Baixo impacto segundo a política didática,fictícia
6,FIC-007,pintura_corredor,Pintura descascada sem umidade ou risco no cor...,0,0,10,25,0,0,Baixo impacto segundo a política didática,fictícia
7,FIC-008,pintura_corredor,Pintura descascada sem umidade ou risco no cor...,0,0,20,30,0,0,Baixo impacto segundo a política didática,fictícia


,registros
prioridade,
0,57
1,63


## 4. Separação por famílias e ajuste usando apenas o treino

X contém os cinco campos especificados; y é `prioridade`. O primeiro fold de `StratifiedGroupKFold` de cinco folds, embaralhado com semente 42, fixa a divisão externa em aproximadamente 80%/20%. A estratificação procura preservar as classes sem separar famílias; famílias inteiras tornam a proporção aproximada. Não selecionamos a partição por acurácia.

A profundidade `[1, 2, 3, 4, 5, 6, None]` é ajustada exclusivamente nos dados de treino por validação cruzada de quatro folds também agrupada por família. F2 favorece sensibilidade da classe prioritária, mas penaliza falsos positivos. Em empate vence a primeira profundidade da lista, mais simples. O conjunto de teste permanece reservado até a próxima seção.

In [6]:

def treinar(df):
    X, y, grupos = df[FEATURES], df["prioridade"], df["familia_cenario"]
    # Primeira partição pré-fixada; nenhuma escolha por desempenho no teste.
    externo = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMENTE)
    indices_treino, indices_teste = next(externo.split(X, y, grupos))
    assert not set(grupos.iloc[indices_treino]) & set(grupos.iloc[indices_teste])
    X_treino, y_treino = X.iloc[indices_treino], y.iloc[indices_treino]
    X_teste, y_teste = X.iloc[indices_teste], y.iloc[indices_teste]
    interno = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEMENTE)
    busca = GridSearchCV(DecisionTreeClassifier(random_state=SEMENTE),
        {"max_depth": [1, 2, 3, 4, 5, 6, None]}, cv=interno,
        scoring={"f2": make_scorer(fbeta_score, beta=2, zero_division=0),
                 "sensibilidade": make_scorer(recall_score, zero_division=0),
                 "precisao": make_scorer(precision_score, zero_division=0)},
        refit="f2", error_score="raise", n_jobs=1)
    # F2 dá peso maior à sensibilidade. Empate: primeiro candidato, árvore menos profunda.
    busca.fit(X_treino, y_treino, groups=grupos.iloc[indices_treino])
    arvore = busca.best_estimator_
    baseline = DummyClassifier(strategy="most_frequent")
    baseline.fit(X_treino, y_treino)
    return arvore, baseline, busca, indices_treino, indices_teste


arvore, baseline, busca, indices_treino, indices_teste = treinar(df)
print("X:", FEATURES, "| y: prioridade")
print("Treino:", len(indices_treino), "| teste:", len(indices_teste))
print("Famílias compartilhadas:", set(df.iloc[indices_treino].familia_cenario) & set(df.iloc[indices_teste].familia_cenario))
print("Profundidade selecionada:", busca.best_params_["max_depth"])
display(pd.DataFrame({"profundidade": [p["max_depth"] for p in busca.cv_results_["params"]], "F2_medio_CV_treino": busca.cv_results_["mean_test_f2"], "sensibilidade_CV_treino": busca.cv_results_["mean_test_sensibilidade"], "precisao_CV_treino": busca.cv_results_["mean_test_precisao"]}))

X: ['risco_seguranca', 'servico_essencial_interrompido', 'pessoas_afetadas', 'dias_aguardando', 'recorrente'] | y: prioridade
Treino: 96 | teste: 24
Famílias compartilhadas: set()
Profundidade selecionada: 6


,profundidade,F2_medio_CV_treino,sensibilidade_CV_treino,precisao_CV_treino
0,1.0,0.288462,0.250000,0.750000
1,2.0,0.623434,0.576923,0.950000
2,3.0,0.637189,0.599359,0.977273
3,4.0,0.803429,0.772436,0.980769
4,5.0,0.952621,0.980769,0.866071
5,6.0,1.000000,1.000000,1.000000
6,NaN,1.000000,1.000000,1.000000


## 5. Avaliação única no teste reservado e comparação

Sensibilidade prioritária = VP/(VP+FN); precisão prioritária = VP/(VP+FP); acurácia = acertos/total. Nas matrizes, linhas são classes reais e colunas são previstas, ambas na ordem `[0, 1]`.

O baseline prevê sempre a classe mais frequente **do treino**, sem consultar a maioria do teste. Se não houver previsões positivas, a precisão é registrada como zero (`zero_division=0`), não como evidência de segurança.

In [7]:

def calcular_metricas(y, predito):
    return {"sensibilidade_prioritaria": float(recall_score(y, predito, pos_label=1, zero_division=0)),
            "precisao_prioritaria": float(precision_score(y, predito, pos_label=1, zero_division=0)),
            "acuracia": float(accuracy_score(y, predito)),
            "matriz_confusao": confusion_matrix(y, predito, labels=[0,1]).tolist(),
            "ordem_classes_matriz": [0,1], "linhas": "real", "colunas": "prevista"}


def avaliar(df, arvore, baseline, busca, indices_treino, indices_teste):
    teste = df.iloc[indices_teste].copy()
    teste["previsto_arvore"] = arvore.predict(teste[FEATURES])
    teste["previsto_baseline"] = baseline.predict(teste[FEATURES])
    fn = teste.loc[(teste.prioridade == 1) & (teste.previsto_arvore == 0),
                   ["id", "familia_cenario", "descricao", *FEATURES, "prioridade", "previsto_arvore"]]
    proporcoes = {}
    for nome, ids in [("base", df.index), ("treino", indices_treino), ("teste", indices_teste)]:
        conjunto = df.loc[ids]
        proporcoes[nome] = {"registros": len(conjunto), "familias": int(conjunto.familia_cenario.nunique()),
            "classes": {str(k): int(v) for k,v in conjunto.prioridade.value_counts().sort_index().items()},
            "proporcao_prioritaria": float(conjunto.prioridade.mean())}
    cv = []
    for pos, parametros in enumerate(busca.cv_results_["params"]):
        cv.append({"max_depth": parametros["max_depth"],
                   "f2_medio": float(busca.cv_results_["mean_test_f2"][pos]),
                   "sensibilidade_media": float(busca.cv_results_["mean_test_sensibilidade"][pos]),
                   "precisao_media": float(busca.cv_results_["mean_test_precisao"][pos])})
    metricas = {"origem": ORIGEM, "semente": SEMENTE, "avaliacao_em": "Google Colab" if ("google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ) else "Ambiente local (Python/nbclient)",
        "politica_rotulos": POLITICA, "divisao": proporcoes,
        "separacao_grupos": "StratifiedGroupKFold 5 folds, primeira partição fixa; famílias disjuntas",
        "ajuste": {"metodo": "GridSearchCV com StratifiedGroupKFold 4 folds exclusivamente no treino", "criterio": "F2 da classe 1; empate favorece menor profundidade", "melhor_max_depth": busca.best_params_["max_depth"], "profundidade_aprendida": arvore.get_depth(), "validacao_cruzada": cv},
        "arvore": calcular_metricas(teste.prioridade, teste.previsto_arvore),
        "baseline_maioria": calcular_metricas(teste.prioridade, teste.previsto_baseline),
        "classe_majoritaria_treino": int(baseline.predict(df.iloc[indices_teste][FEATURES])[0]),
        "falsos_negativos": fn.to_dict(orient="records"),
        "versoes": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "scikit_learn": sklearn.__version__, "joblib": joblib.__version__},
        "limites": ["Teste pequeno e fictício: métricas não estimam desempenho em instituições reais.", "Rótulos derivados de uma política explícita: o modelo aproxima regras escolhidas, sem descoberta de risco real.", "Famílias separadas reduzem vazamento de variações; padrões de atributos podem se repetir entre famílias.", "Ausência de falsos negativos neste teste, se observada, não garante segurança fora da amostra.", "Nenhuma previsão substitui inspeção técnica, resposta a emergências ou decisão do responsável."]}
    return metricas, teste


metricas, teste = avaliar(df, arvore, baseline, busca, indices_treino, indices_teste)
print("Execução:", metricas["avaliacao_em"])
display(pd.DataFrame(metricas["divisao"]).T)
display(pd.DataFrame({"Árvore": {k:v for k,v in metricas["arvore"].items() if isinstance(v,float)}, "Baseline (maioria)": {k:v for k,v in metricas["baseline_maioria"].items() if isinstance(v,float)}}).T)
for nome in ["arvore", "baseline_maioria"]:
    print(nome, "— linhas reais, colunas previstas")
    display(pd.DataFrame(metricas[nome]["matriz_confusao"], index=["Real normal", "Real prioritário"], columns=["Previsto normal", "Previsto prioritário"]))
if metricas["falsos_negativos"]:
    display(pd.DataFrame(metricas["falsos_negativos"]))
else:
    print("Nenhum falso negativo observado nos", int((teste.prioridade == 1).sum()), "chamados prioritários do teste. Isso não garante ausência de erros fora desta amostra fictícia.")
display(teste[["id", "familia_cenario", "prioridade", "previsto_arvore", "previsto_baseline"]])

Execução: Ambiente local (Python/nbclient)


,registros,familias,classes,proporcao_prioritaria
base,120,30,"{'0': 57, '1': 63}",0.525
treino,96,24,"{'0': 47, '1': 49}",0.510417
teste,24,6,"{'0': 10, '1': 14}",0.583333


,sensibilidade_prioritaria,precisao_prioritaria,acuracia
Árvore,1.0,1.000000,1.000000
Baseline (maioria),1.0,0.583333,0.583333


arvore — linhas reais, colunas previstas


,Previsto normal,Previsto prioritário
Real normal,10,0
Real prioritário,0,14


baseline_maioria — linhas reais, colunas previstas


,Previsto normal,Previsto prioritário
Real normal,0,10
Real prioritário,0,14


Nenhum falso negativo observado nos 14 chamados prioritários do teste. Isso não garante ausência de erros fora desta amostra fictícia.


,id,familia_cenario,prioridade,previsto_arvore,previsto_baseline
12,FIC-013,cadeira_estofado,0,0,1
13,FIC-014,cadeira_estofado,0,0,1
14,FIC-015,cadeira_estofado,0,0,1
15,FIC-016,cadeira_estofado,0,0,1
24,FIC-025,torneira_auxiliar,0,0,1
25,FIC-026,torneira_auxiliar,0,0,1
26,FIC-027,torneira_auxiliar,0,0,1
27,FIC-028,torneira_auxiliar,0,0,1
68,FIC-069,piso_escorregadio,1,1,1
69,FIC-070,piso_escorregadio,1,1,1


## 6. Exportação da mesma árvore aprendida

Exportamos a árvore treinada nos **96 registros de treino**, sem retreinar no teste. `modelo/arvore.json` registra ordem dos campos, classes, nós, limiares, filhos, contagens e classe em cada nó. A travessia vai para a esquerda quando o valor é menor ou igual ao limiar. O Flask e o navegador usam esse JSON; nenhuma classificação em produção chama a função que gerou os rótulos.

O `.joblib` serve à comparação com scikit-learn nos testes. Só carregue um joblib de origem confiável. Os limites e a validação de entradas são tratados pela aplicação. Nenhuma credencial pertence ao modelo.

In [8]:

def exportar(raiz, df, arvore, baseline, busca, indices_treino, indices_teste, metricas, teste):
    raiz = Path(raiz)
    for pasta in ["dados", "modelo", "relatorios"]:
        (raiz / pasta).mkdir(parents=True, exist_ok=True)
    csv = df.to_csv(index=False, lineterminator="\n")
    (raiz / "dados/chamados_ficticios.csv").write_text(csv, encoding="utf-8")
    arv = arvore.tree_
    nos = []
    for i in range(arv.node_count):
        # O valor do sklearn pode ser proporção; escalamos pelas amostras ponderadas.
        contagens = (arv.value[i][0] / arv.value[i][0].sum() * arv.weighted_n_node_samples[i]).tolist()
        nos.append({"id": i, "feature_index": int(arv.feature[i]),
            "threshold": float(arv.threshold[i]), "left": int(arv.children_left[i]),
            "right": int(arv.children_right[i]), "counts": [float(v) for v in contagens],
            "prediction": int(arvore.classes_[int(np.argmax(arv.value[i][0]))])})
    modelo = {"schema_version": 1, "model_type": "DecisionTreeClassifier", "root": 0,
        "feature_order": FEATURES, "classes": [int(v) for v in arvore.classes_],
        "class_labels": {"0": "Normal", "1": "Prioritário"},
        "comparison": "left if float32(value) <= threshold; otherwise right",
        "nodes": nos, "metadata": {"origem": ORIGEM, "autor": "Guilherme Costa",
            "semente": SEMENTE, "max_depth": busca.best_params_["max_depth"],
            "sklearn_version": sklearn.__version__, "data_sha256": hashlib.sha256(csv.encode()).hexdigest(),
            "uso": "Demonstração acadêmica com revisão humana obrigatória", "limites_entrada": {"pessoas_afetadas": [0, 9007199254740991], "dias_aguardando": [0, 9007199254740991]}}}
    (raiz / "modelo/arvore.json").write_text(json.dumps(modelo, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
    (raiz / "modelo/regras_arvore.txt").write_text(export_text(arvore, feature_names=FEATURES), encoding="utf-8")
    joblib.dump(arvore, raiz / "modelo/arvore.joblib")
    divisao = df[["id", "familia_cenario", "prioridade"]].copy()
    divisao["conjunto"] = "treino"
    divisao.loc[indices_teste, "conjunto"] = "teste"
    divisao.to_csv(raiz / "dados/divisao_treino_teste.csv", index=False)
    teste.to_csv(raiz / "relatorios/predicoes_teste.csv", index=False)
    (raiz / "relatorios/metricas.json").write_text(json.dumps(metricas, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
    return modelo


raiz_saida = Path.cwd()
modelo = exportar(raiz_saida, df, arvore, baseline, busca, indices_treino, indices_teste, metricas, teste)
print("Arquivos gerados em:", raiz_saida)
print("Ordem:", modelo["feature_order"], "| classes:", modelo["classes"])
print("Nós:", len(modelo["nodes"]))
print(export_text(arvore, feature_names=FEATURES))

Arquivos gerados em: /workspace/work/priorizacao-chamados-manutencao
Ordem: ['risco_seguranca', 'servico_essencial_interrompido', 'pessoas_afetadas', 'dias_aguardando', 'recorrente'] | classes: [0, 1]
Nós: 15
|--- servico_essencial_interrompido <= 0.50
|   |--- risco_seguranca <= 0.50
|   |   |--- pessoas_afetadas <= 39.50
|   |   |   |--- dias_aguardando <= 13.50
|   |   |   |   |--- class: 0
|   |   |   |--- dias_aguardando >  13.50
|   |   |   |   |--- recorrente <= 0.50
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- recorrente >  0.50
|   |   |   |   |   |--- pessoas_afetadas <= 9.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |   |--- pessoas_afetadas >  9.50
|   |   |   |   |   |   |--- class: 1
|   |   |--- pessoas_afetadas >  39.50
|   |   |   |--- dias_aguardando <= 6.50
|   |   |   |   |--- class: 0
|   |   |   |--- dias_aguardando >  6.50
|   |   |   |   |--- class: 1
|   |--- risco_seguranca >  0.50
|   |   |--- class: 1
|--- servico_essencial_interrompido

## 7. Verificação da exportação e três cenários fictícios

Esta checagem verifica o artefato JSON contra o scikit-learn em toda a base. Os testes do repositório comparam adicionalmente Flask e JavaScript e verificam formulários e entradas inválidas.

In [9]:
def prever_json(valores, modelo):
    indice = modelo["root"]
    while modelo["nodes"][indice]["left"] != -1:
        no = modelo["nodes"][indice]
        indice = no["left"] if np.float32(valores[no["feature_index"]]) <= no["threshold"] else no["right"]
    return modelo["nodes"][indice]["prediction"]
previsto_json = [prever_json(linha, modelo) for linha in df[FEATURES].to_numpy()]
assert previsto_json == arvore.predict(df[FEATURES]).tolist()
print("Paridade JSON/scikit-learn confirmada em", len(df), "registros.")
exemplos = pd.DataFrame([
    [0, 0, 0, 2, 0],
    [0, 1, 200, 0, 0],
    [1, 0, 1, 0, 0],
], columns=FEATURES, index=["Maçaneta frouxa em sala desocupada", "Falta de água na escola", "Fiação exposta com pouca circulação"])
exemplos["prioridade_sugerida"] = arvore.predict(exemplos[FEATURES])
display(exemplos)
assert exemplos.prioridade_sugerida.tolist() == [0, 1, 1]

Paridade JSON/scikit-learn confirmada em 120 registros.


,risco_seguranca,servico_essencial_interrompido,pessoas_afetadas,dias_aguardando,recorrente,prioridade_sugerida
Maçaneta frouxa em sala desocupada,0,0,0,2,0,0
Falta de água na escola,0,1,200,0,0,1
Fiação exposta com pouca circulação,1,0,1,0,0,1


## 8. Interpretação e limites

A base é pequena, artificial e rotulada por regras explícitas; a árvore aproxima essas regras. Uma acurácia de 100% neste teste não demonstra capacidade de prever urgências reais. O baseline majoritário também detecta todos os prioritários neste recorte, ao custo de classificar os normais como prioritários. A comparação deve considerar falsos positivos, falsos negativos e a capacidade real da equipe.

Amostra, limiares e descrições não representam todas as instituições. Classificação binária não estima gravidade médica, probabilidade de acidente ou prazo seguro. Uma descrição pode conter fatos não representados pelos cinco campos. O responsável precisa verificar inconsistências, avaliar o local e revisar a sugestão antes de decidir. Situações de emergência devem seguir o procedimento institucional.

## 9. Plano para dados reais, ainda não realizado

1. Formalizar com uma instituição parceira a finalidade de apoiar a triagem, os responsáveis, as permissões de acesso, o prazo de retenção e os critérios de urgência. Não há instituição parceira ou autorização real nesta entrega.
2. Coletar prospectivamente chamados de períodos e áreas variados, incluindo casos simples, raros e limítrofes. Registrar momento da abertura, atributos disponíveis naquele momento e família/evento; evitar dados posteriores ao atendimento como entrada.
3. Remover nomes, telefones, matrículas, endereços pessoais e outros identificadores de descrições e anexos. Usar identificador aleatório e local em granularidade suficiente para manutenção, com acesso restrito. Excluir imagens identificáveis; pseudonimização isolada não garante anonimato. Não publicar a base real no GitHub.
4. Dois responsáveis por manutenção rotulam independentemente, com manual de critérios e opção “incerto”. Resolver divergências por revisão técnica documentada; medir concordância e discutir o custo de falsos negativos. Não usar o próprio modelo como fonte automática de verdade.
5. Manter registros do mesmo evento/família juntos; reservar também período futuro ou instituição externa para avaliar generalização. Ajustar hiperparâmetros somente no treino. Acompanhar sensibilidade, precisão, matriz de confusão, exemplos de erros e intervalos de incerteza.
6. Fazer piloto supervisionado, comparar com triagem humana, monitorar mudanças e revisar os critérios com a instituição. A IA generativa opcional pode sugerir campos, mas valores ausentes permanecem desconhecidos, exigem confirmação e a classe final continua vindo da árvore.

## 10. Artefatos e entrega

O notebook gera `dados/chamados_ficticios.csv`, a divisão por ID/família, `modelo/arvore.json`, `modelo/arvore.joblib`, as regras legíveis, as métricas e todas as previsões do teste. No Colab, esses arquivos aparecem no painel **Arquivos**; o código abaixo cria um ZIP opcional desses resultados. A entrega completa do projeto inclui também frontend, Flask, dependências, testes e README fornecidos separadamente.

In [10]:
from zipfile import ZipFile, ZIP_DEFLATED
artefatos = ["dados/chamados_ficticios.csv", "dados/divisao_treino_teste.csv", "modelo/arvore.json", "modelo/arvore.joblib", "modelo/regras_arvore.txt", "relatorios/metricas.json", "relatorios/predicoes_teste.csv"]
with ZipFile("resultados_notebook.zip", "w", compression=ZIP_DEFLATED) as pacote:
    for caminho in artefatos:
        assert Path(caminho).is_file(), caminho
        pacote.write(caminho)
print("Concluído: resultados_notebook.zip criado. Dados inteiramente fictícios; revisão humana obrigatória.")

Concluído: resultados_notebook.zip criado. Dados inteiramente fictícios; revisão humana obrigatória.
